# 15-phase5-task-scale

**neuron Phase 5** — Phase 4 시나리오 B 의 빠른 재검증. max_steps 만 1500→5000 으로 늘려
per-channel α 의 표현력 우위가 더 긴 학습에서 발현되는지 확인.

- **Phase 4 결과** (PR #48): per-channel vs scalar Δ=-0.0015 (σ 0.007 보다 작음, 통계적 무의미)
- **본 Phase 의 가설**:
  1. **장기 학습 시 per-channel 우위 발현** — 5000step 에서 per-channel < scalar 명확? (Δ > σ)
  2. **partial-dead 증가** — 더 긴 학습으로 채널별 자율 억제 더 적극적 진화?
  3. **scalar 도 장기 이익?** — 둘 다 saturation 가능성 점검
- **설계**: 2 × 3 sweep (α_type × seed), max_steps=5000 (3.3×), 나머지 Phase 4 동일.
- **데이터**: TinyShakespeare (char-LM)
- **작성일**: 2026-05-25
- **연관**: Issue [#49](https://github.com/EinSofINTEREST/GraphLM/issues/49) / Phase 4 baseline PR [#48](https://github.com/EinSofINTEREST/GraphLM/pull/48)

## 1. 환경 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import NeuronConfig, NeuronGrowingDecoder, add_attn_smooth_start
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정 — max_steps=5000 만 변경

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase5"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
ALPHA_TYPES = ["scalar", "per_channel"]
ALPHA_INIT = 0.10

BATCH_SIZE = 16
BLOCK_SIZE = 128
BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_layers=4, n_init_attn=1)
TRAIN = dict(
    lr=3e-4,
    max_steps=5000,  # ← Phase 4 의 1500 에서 3.3×
    max_total_attn=8,  # Phase 4 와 동일 (fair 비교)
    trigger_window=100,
    trigger_epsilon=0.04,
    trigger_cooldown=150,
    trigger_min_history=100,
)
DEAD_THRESHOLD = 0.05
PARTIAL_DEAD_THRESH = 0.80

# Phase 4 baseline (PR #48 결과 — 1500step)
PHASE4_SCALAR_MEAN = 1.7763
PHASE4_SCALAR_SIGMA = 0.0070
PHASE4_PERCH_MEAN = 1.7778
PHASE4_PERCH_SIGMA = 0.0063
GRAPHLM_PHASE1_4L_FINAL_LOSS = 1.7871

print(f"SEEDS        = {SEEDS}")
print(f"ALPHA_TYPES  = {ALPHA_TYPES}")
print(f"ALPHA_INIT   = {ALPHA_INIT}")
print(f"max_steps    = {TRAIN['max_steps']} (Phase 4 의 1500 에서 3.3×)")
print(f"총 run       = {len(SEEDS) * len(ALPHA_TYPES)} (GPU 약 25분 예상)")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 2 × 3 sweep 학습 (5000step)

코드는 Phase 4 와 동일 — max_steps 만 다름. loss 를 step 별로 저장해 §8 에서 curve 비교.

In [ ]:
runs = {}
for alpha_type in ALPHA_TYPES:
    per_channel = alpha_type == "per_channel"
    cfg = NeuronConfig(
        vocab_size=tokenizer.vocab_size,
        max_seq_len=BLOCK_SIZE,
        alpha_per_channel=per_channel,
        **BACKBONE,
    )
    for seed in SEEDS:
        print(f"--- alpha_type={alpha_type}, seed={seed} ---")
        set_seed(seed)
        model = NeuronGrowingDecoder(cfg).to(DEVICE)
        data_iter = iter_random_batches(
            dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
        )
        trigger = PlateauTrigger(
            window=TRAIN["trigger_window"],
            epsilon=TRAIN["trigger_epsilon"],
            cooldown=TRAIN["trigger_cooldown"],
            min_history=TRAIN["trigger_min_history"],
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

        losses = []
        grow_events = []
        next_block_to_grow = 0
        V = cfg.vocab_size
        model.train()
        for step in range(1, TRAIN["max_steps"] + 1):
            x, y = next(data_iter)
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            if (
                trigger.update(loss.item())
                and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]
            ):
                target_block = next_block_to_grow
                add_attn_smooth_start(model, target_block, alpha_init=ALPHA_INIT)
                block = model.blocks[target_block]
                new_params = (
                    list(block.attn_lns[-1].parameters())
                    + list(block.attns[-1].parameters())
                    + [block.attn_alphas[-1]]
                )
                optimizer.add_param_group({"params": new_params, "lr": TRAIN["lr"]})
                grow_events.append((step, target_block, sum(model.n_attn_per_block)))
                next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

        final_alphas = [
            (bi, ai, alpha_param.detach().cpu().clone())
            for bi, block in enumerate(model.blocks)
            for ai, alpha_param in enumerate(block.attn_alphas)
        ]
        runs[(alpha_type, seed)] = {
            "losses": losses,
            "grow_events": grow_events,
            "final_alphas": final_alphas,
        }
        n_last = min(100, len(losses))
        final_loss = sum(losses[-n_last:]) / n_last if n_last > 0 else 0.0
        print(f"  done: final_loss(last 100 avg)={final_loss:.4f}, n_added={len(grow_events)}")

        del model, optimizer, trigger, data_iter
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 — α_type × seed (5000step)

In [ ]:
import math
import statistics

print(
    f"{'α_type':>11}  {'seed':>5}  {'n_added':>7}  {'attn-dead%':>10}  "
    f"{'ch-dead%':>9}  {'mean|α|':>8}  {'final_loss':>11}"
)
print("-" * 80)
table = {}
for alpha_type in ALPHA_TYPES:
    table[alpha_type] = {}
    for seed in SEEDS:
        r = runs[(alpha_type, seed)]
        added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
        n_added = len(added)
        if n_added == 0:
            attn_dead_pct = math.nan
            ch_dead_pct = math.nan
            mean_abs = math.nan
        elif alpha_type == "scalar":
            abs_vals = [a.abs().item() for _, _, a in added]
            attn_dead_pct = sum(1 for v in abs_vals if v < DEAD_THRESHOLD) / n_added
            ch_dead_pct = attn_dead_pct
            mean_abs = statistics.mean(abs_vals)
        else:
            ch_dead_per_attn = []
            attn_dead_count = 0
            all_abs = []
            for _, _, a in added:
                abs_v = a.abs()
                ch_dead = (abs_v < DEAD_THRESHOLD).float().mean().item()
                ch_dead_per_attn.append(ch_dead)
                if ch_dead >= PARTIAL_DEAD_THRESH:
                    attn_dead_count += 1
                all_abs.extend(abs_v.tolist())
            attn_dead_pct = attn_dead_count / n_added
            ch_dead_pct = statistics.mean(ch_dead_per_attn)
            mean_abs = statistics.mean(all_abs)

        n_last = min(100, len(r["losses"]))
        final_loss = sum(r["losses"][-n_last:]) / n_last if n_last > 0 else 0.0
        table[alpha_type][seed] = dict(
            n_added=n_added,
            attn_dead_pct=attn_dead_pct,
            ch_dead_pct=ch_dead_pct,
            mean_abs=mean_abs,
            final_loss=final_loss,
        )
        if n_added == 0:
            attn_str, ch_str, mean_str = "    n/a ", "    n/a ", "    n/a "
        else:
            attn_str = f"{attn_dead_pct:>10.1%}"
            ch_str = f"{ch_dead_pct:>9.1%}"
            mean_str = f"{mean_abs:>8.4f}"
        print(
            f"{alpha_type:>11}  {seed:>5}  {n_added:>7d}  {attn_str}  {ch_str}  "
            f"{mean_str}  {final_loss:>11.4f}"
        )

## 6. Phase 4 (1500step) 대비 진행도 + 통계 판정

In [ ]:
# 본 Phase 5 mean ± σ
agg5 = {}
for alpha_type in ALPHA_TYPES:
    fls = [table[alpha_type][s]["final_loss"] for s in SEEDS]
    agg5[alpha_type] = dict(
        mean=statistics.mean(fls),
        sigma=statistics.stdev(fls) if len(fls) > 1 else 0,
    )

print("=== Phase 4 (1500step) → Phase 5 (5000step) 진행도 ===")
print(
    f"  scalar      : 1500step {PHASE4_SCALAR_MEAN:.4f} ± {PHASE4_SCALAR_SIGMA:.4f} "
    f"→ 5000step {agg5['scalar']['mean']:.4f} ± {agg5['scalar']['sigma']:.4f} "
    f"(Δ {agg5['scalar']['mean'] - PHASE4_SCALAR_MEAN:+.4f})"
)
print(
    f"  per_channel : 1500step {PHASE4_PERCH_MEAN:.4f} ± {PHASE4_PERCH_SIGMA:.4f} "
    f"→ 5000step {agg5['per_channel']['mean']:.4f} ± {agg5['per_channel']['sigma']:.4f} "
    f"(Δ {agg5['per_channel']['mean'] - PHASE4_PERCH_MEAN:+.4f})"
)

print()
print("=== Phase 5 5000step: scalar vs per-channel 직접 비교 ===")
diff = agg5["scalar"]["mean"] - agg5["per_channel"]["mean"]
larger_sigma = max(agg5["scalar"]["sigma"], agg5["per_channel"]["sigma"])
print(f"  scalar      mean ± σ : {agg5['scalar']['mean']:.4f} ± {agg5['scalar']['sigma']:.4f}")
print(
    f"  per_channel mean ± σ : {agg5['per_channel']['mean']:.4f} ± {agg5['per_channel']['sigma']:.4f}"
)
print(f"  차이 (scalar - per_ch): {diff:+.4f}")
print(f"  max σ                 : {larger_sigma:.4f}")
ratio = abs(diff) / larger_sigma if larger_sigma > 0 else float("inf")
print(f"  |Δ| / σ                : {ratio:.2f}")
if ratio < 1.0:
    verdict = "⚖️ 통계적으로 무의미 (|Δ| < σ) — Phase 4 와 동일 결론"
elif diff > 0:
    # diff = scalar.mean - per_channel.mean > 0 → per_channel 의 loss 가 더 낮음 → per-channel 우위
    verdict = "✅ per-channel 명확 우위 (|Δ| > σ) — 장기 학습에서 발현"
else:
    # diff < 0 → scalar 의 loss 가 더 낮음 → scalar 우위
    verdict = "❌ scalar 명확 우위 (|Δ| > σ) — per-channel 의 추가 자유도가 noise"
print(f"  verdict               : {verdict}")
n_pc_better = sum(
    1 for s in SEEDS if table["per_channel"][s]["final_loss"] < table["scalar"][s]["final_loss"]
)
print(f"  per-channel < scalar  : {n_pc_better}/{len(SEEDS)} 시드")

## 7. partial-dead 진화 (Phase 4 → Phase 5)

In [ ]:
# Phase 4 (1500step) 의 per-channel ch-dead 범위: 0.0% ~ 1.4%
# Phase 5 (5000step) 에서 더 적극적으로 증가하는지
print("=== per-channel ch-dead % 진화 ===")
print("  Phase 4 (1500step): 0.0% ~ 1.4% (PR #48)")
ch_deads_5 = [table["per_channel"][s]["ch_dead_pct"] for s in SEEDS]
print(f"  Phase 5 (5000step): {min(ch_deads_5):.1%} ~ {max(ch_deads_5):.1%}")
print(f"  mean ch-dead     : {statistics.mean(ch_deads_5):.1%}")

print()
print("=== seed 별 per-channel added attn 의 channel 분포 (5000step) ===")
print(
    f"{'seed':>5}  {'added':>5}  {'block':>5}  {'min|α|':>8}  {'mean|α|':>8}  "
    f"{'max|α|':>8}  {'dead%':>6}"
)
print("-" * 65)
for seed in SEEDS:
    r = runs[("per_channel", seed)]
    added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
    for idx, (bi, _ai, a) in enumerate(added):
        absv = a.abs()
        dead_frac = (absv < DEAD_THRESHOLD).float().mean().item()
        print(
            f"{seed:>5}  {idx:>5}  {bi:>5}  {absv.min().item():>8.4f}  "
            f"{absv.mean().item():>8.4f}  {absv.max().item():>8.4f}  {dead_frac:>6.1%}"
        )

## 8. 시각화 — loss curve + final_loss 비교

(a) loss curve smoothed (윈도우 100) — scalar vs per_channel 의 발산/수렴 step 구간 확인.
(b) final_loss bar + Phase 4 baseline 비교.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1])

# (a) loss curve — α_type 별 mean ± shade (3 seed)
ax1 = fig.add_subplot(gs[0, 0])
window = 100  # smoothing window
colors = {"scalar": "#1f77b4", "per_channel": "#ff7f0e"}
for alpha_type in ALPHA_TYPES:
    seed_curves = []
    for seed in SEEDS:
        losses = runs[(alpha_type, seed)]["losses"]
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        seed_curves.append(smoothed)
    arr = np.array(seed_curves)
    steps = np.arange(window - 1, window - 1 + arr.shape[1])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0, ddof=1)
    ax1.plot(steps, mean, color=colors[alpha_type], label=alpha_type, lw=1.5)
    ax1.fill_between(steps, mean - std, mean + std, color=colors[alpha_type], alpha=0.2)

ax1.axhline(
    PHASE4_SCALAR_MEAN,
    color="#1f77b4",
    linestyle=":",
    lw=1,
    alpha=0.7,
    label=f"Phase 4 scalar 1500step ({PHASE4_SCALAR_MEAN})",
)
ax1.axhline(
    PHASE4_PERCH_MEAN,
    color="#ff7f0e",
    linestyle=":",
    lw=1,
    alpha=0.7,
    label=f"Phase 4 per_channel 1500step ({PHASE4_PERCH_MEAN})",
)
ax1.axhline(
    GRAPHLM_PHASE1_4L_FINAL_LOSS,
    color="red",
    linestyle="--",
    lw=1,
    alpha=0.7,
    label=f"GraphLM 4L baseline ({GRAPHLM_PHASE1_4L_FINAL_LOSS})",
)
ax1.set_xlabel("step")
ax1.set_ylabel(f"loss (smoothed, window={window})")
ax1.set_title("loss curves — scalar vs per_channel (mean ± σ across 3 seeds, 5000 steps)")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(alpha=0.3)

# (b) final_loss bar
ax2 = fig.add_subplot(gs[1, 0])
width = 0.35
x_seeds = list(range(len(SEEDS)))
all_final = [table[t][s]["final_loss"] for t in ALPHA_TYPES for s in SEEDS]
for i, alpha_type in enumerate(ALPHA_TYPES):
    offsets = [x + (i - 0.5) * width for x in x_seeds]
    vals = [table[alpha_type][s]["final_loss"] for s in SEEDS]
    ax2.bar(offsets, vals, width=width, color=colors[alpha_type], label=alpha_type, alpha=0.85)
ax2.axhline(
    PHASE4_SCALAR_MEAN, color="#1f77b4", linestyle=":", lw=1, alpha=0.7, label="Phase 4 scalar mean"
)
ax2.axhline(
    PHASE4_PERCH_MEAN,
    color="#ff7f0e",
    linestyle=":",
    lw=1,
    alpha=0.7,
    label="Phase 4 per_channel mean",
)
ax2.set_xlabel("seed")
ax2.set_xticks(x_seeds)
ax2.set_xticklabels([str(s) for s in SEEDS])
ax2.set_ylabel("final loss (last 100 avg)")
ax2.set_title("Phase 5 final_loss vs Phase 4 baselines")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(alpha=0.3, axis="y")
ymin_d = min(min(all_final), PHASE4_SCALAR_MEAN, PHASE4_PERCH_MEAN)
ymax_d = max(max(all_final), PHASE4_SCALAR_MEAN, PHASE4_PERCH_MEAN)
margin = max(0.02, (ymax_d - ymin_d) * 0.15)
ax2.set_ylim(ymin_d - margin, ymax_d + margin)

fig.tight_layout()
fig.savefig(OUT_DIR / "task_scale.png", dpi=120)
plt.show()

## 결과 요약 / Phase 6 권장 방향

이 노트북에서 확인할 것:
- §6 의 |Δ|/σ verdict — 1.0 미만이면 시나리오 B 유지, 초과 시 시나리오 A
- §7 의 ch-dead 진화 — 1.4% → 더 증가? saturation?
- §8 (a) loss curve — scalar / per_channel 발산 step 구간 (분기점 확인)

**판정 시나리오**:
- **A. per-channel 명확 우위 (|Δ| > σ)** — 잠재력 확인됨
  - Phase 6: 더 큰 task (WikiText-103 subset, BookCorpus 등) 일관 검증
- **B. 여전히 동등 (|Δ| < σ)** — per-channel 표현력은 본 paradigm 에서 limit
  - Phase 6: relative-position 함수형 α (Notion 의 기존 Phase 5+ 후보, 현재 Phase 6+)
- **C. scalar 우위** (드물지만)
  - per-channel 의 추가 자유도가 noise — 회귀, 다른 axis (예: full matrix α, RoPE 변형)

**참고 자료**:
- Phase 4 결과 정리 (Notion): https://www.notion.so/36be8b70b7aa8117a3b9d7671df5f41b
- Phase 5 계획 (Notion): https://www.notion.so/36be8b70b7aa812cae39d41a3660f0e7
- Phase 6+ 후보 (Notion): https://www.notion.so/36be8b70b7aa8142bebdc9706c64ed52